In [1]:
from pathlib import Path

from clearml import Dataset
import pandas as pd
from tqdm.notebook import tqdm

# Params

In [2]:
freq = "15min"

# Total Power

In [3]:
data_path = Dataset.get(
    dataset_project="ForeSightNEXT/Electric Load Forecasting",
    dataset_name="household-1235",
).get_local_copy()

raw_data_path = Path(data_path, "power.csv")
resampled_data_path = Path(f"../data/resampled-{freq}/total_power.csv")

In [4]:
if resampled_data_path.exists() and resampled_data_path.is_file():
    print("Resampled File already exists")
    resampled_data = pd.read_csv(
        resampled_data_path,
        index_col="_time",
        parse_dates=True,
        date_format="ISO8601",
    )
else:
    data = pd.read_csv(
        raw_data_path,
        usecols=["_time", "_value"],
        index_col="_time",
        parse_dates=True,
        date_format="ISO8601",
    )
    data.rename(columns={"_value": "total"}, inplace=True)
    
    resampled_data_path.parent.mkdir(exist_ok=True, parents=True)
    resampled_data = data.resample(freq).mean()
    resampled_data.to_csv(resampled_data_path)

display(resampled_data)

Resampled File already exists


,total
_time,
2020-07-28 14:00:00+00:00,300.014388
2020-07-28 14:15:00+00:00,251.965909
2020-07-28 14:30:00+00:00,289.356061
2020-07-28 14:45:00+00:00,230.844828
2020-07-28 15:00:00+00:00,270.125000
...,...
2023-01-30 12:30:00+00:00,750.270000
2023-01-30 12:45:00+00:00,403.076412
2023-01-30 13:00:00+00:00,465.383333


# Devices

In [5]:
shiftable = {
    "DECT210 Waschmaschine": "uri:urn:6a112240-117e-4ec3-a129-5bc90908aedb",    
    "DECT200 Waschmaschine": "uri:urn:4b29c04c920141e8",
    "Smart Switch 6 Spülmaschine": "uri:urn:e91f9319-71af-4ddb-ab7d-fb47b45d69ad",    
    "DECT200 Spülmaschine": "uri:urn:cc256ae649904024",
    # "Smart Switch 6 Thermomix": "uri:urn:5b59894b-805d-4d7c-96c9-8ab42a18c3c7",
}

hh_root = Path("../../../datasets/ForeSight/household-1235/")
ts_dirs = {k: Path(hh_root, fn) for k, fn in shiftable.items()}
ts_dirs

{'DECT210 Waschmaschine': PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:6a112240-117e-4ec3-a129-5bc90908aedb'),
 'DECT200 Waschmaschine': PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:4b29c04c920141e8'),
 'Smart Switch 6 Spülmaschine': PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:e91f9319-71af-4ddb-ab7d-fb47b45d69ad'),
 'DECT200 Spülmaschine': PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:cc256ae649904024')}

In [6]:
for k, td in ts_dirs.items():
    print(k)
    display(list(td.iterdir()))
    print("\n")

DECT210 Waschmaschine


[PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:6a112240-117e-4ec3-a129-5bc90908aedb/socket-on.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:6a112240-117e-4ec3-a129-5bc90908aedb/sensor_160-value.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:6a112240-117e-4ec3-a129-5bc90908aedb/sensor_163-value.csv')]



DECT200 Waschmaschine


[PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:4b29c04c920141e8/temperature.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:4b29c04c920141e8/onOff.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:4b29c04c920141e8/energy.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:4b29c04c920141e8/power.csv')]



Smart Switch 6 Spülmaschine


[PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:e91f9319-71af-4ddb-ab7d-fb47b45d69ad/sensor_582-value.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:e91f9319-71af-4ddb-ab7d-fb47b45d69ad/socket-on.csv')]



DECT200 Spülmaschine


[PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:cc256ae649904024/temperature.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:cc256ae649904024/onOff.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:cc256ae649904024/energy.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:cc256ae649904024/power.csv')]

In [7]:
def clean_cols(thing_name: str, data: pd.DataFrame):
    """ CAUTION: Side effects!
    """
    match thing_name:
        case "DECT210 Waschmaschine":
            data.rename(columns={"sensor_160-value": "power"}, inplace=True)
            data.drop(["sensor_163-value"], axis=1, inplace=True)
        case "Smart Switch 6 Spülmaschine":
            data.rename(columns={"sensor_582-value": "power"}, inplace=True)
    return data  

def read_tables(thing_name: str, freq: str, **kwargs):
    frames = []
    for p in tqdm(
        list(ts_dirs[thing_name].iterdir())# + 
    ):
        if p.suffix == ".csv" and any([
            p.stem == "energy",
            p.stem == "power",
            p.stem.startswith("sensor"),
        ]):
            df = pd.read_csv(
                p,
                usecols=["_time", "_value"],
                index_col="_time",
                parse_dates=True,
                date_format="ISO8601",
                **kwargs,
            )
            df.columns = [p.stem]
            df.astype(float)
            df = df.resample(freq).mean()
            frames.append(df)
    bulk = pd.concat(frames, axis=1)
    bulk = clean_cols(thing_name, bulk)
    return bulk

def cached_resample(thing_name: str, freq=str):
    out_path = Path(f"../data/resampled-{freq}/{thing_name}.csv")
    if out_path.exists() and out_path.is_file():
        print("reading from cache...")
        time_series = pd.read_csv(
            out_path, 
            index_col="_time",
            parse_dates=True,
            date_format="ISO8601"
        )
    else:
        print("resampling...")
        time_series = read_tables(thing_name=thing_name, freq=freq)
        out_path.parent.mkdir(parents=True, exist_ok=True)   
        time_series.to_csv(out_path)

    return time_series

In [8]:
series = {thing_name: cached_resample(thing_name, freq) for thing_name in ts_dirs.keys()}

reading from cache...
reading from cache...
reading from cache...
reading from cache...
